In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as LogNorm
import matplotlib.ticker as ticker
import math
#from lmfit import Model
from scipy.optimize import curve_fit
from scipy.stats import chisquare
from scipy.odr import ODR, Model, RealData
from scipy.signal import savgol_filter

import ROOT
import boost_histogram as bh


In [ ]:
tfile_hist = [f"../histograms/analysis_note/theta_phi_{i}.root" for i in [10.2, 10.4, 10.6] ]
par_list = ['e', 'pip', 'pim']
par_tit = ['e', r'{\pi^+}', r'{\pi^-}']
hBase = 'hThetaPhi_sec'

In [ ]:
import numpy as np

def rebin2d(H, fx, fy):
    nx, ny = H.shape
    assert nx % fx == 0 and ny % fy == 0
    return H.reshape(nx//fx, fx, ny//fy, fy).sum(axis=(1, 3))
def rebin2d_with_errors(H, E, fx, fy):
    H2 = rebin2d(H, fx, fy)
    E2 = np.sqrt(rebin2d(E**2, fx, fy))
    return H2, E2


In [ ]:
#function list
phi_avg = 0
side = 1
parIdx = 0

theta_min_fixed = 0

def trapezoid(x, x0, x1, x2, x3, height):
    return np.piecewise(
        x,
        [x < x0,
         (x >= x0) & (x < x1),
         (x >= x1) & (x < x2),
         (x >= x2) & (x <= x3),
         x > x3],
        [0,
         lambda x: ((x - x0) / (x1 - x0)) * height,
         height,
         lambda x: ((x3 - x) / (x3 - x2)) * height,
         0]
    )

def expo(x, a, b, c, d):
    return a/(x-b) +c
    return a*np.exp(b*x+c)+d
def poly3(x, a, b, c, d):

    return a + b*x + c*x**2 + d*x**3

def getPercentile( arr, nBins, perc ):
    for i in range(nBins):
        if np.sum(arr[:i])/np.sum(arr) > 0.01*perc:
            return i
    return 0

def lorGauss(x, mu, sig1, sig2, A, cutoff):
    switchOver = A/(np.pi)*((sig2)/((cutoff-mu)**2 + sig2**2))
    return np.piecewise(
        x,
        [x < cutoff - switchOver/sig1,
            (x >= cutoff - switchOver/sig1) & (x < cutoff),
         x >= cutoff],
        [
         0,
         lambda x: sig1*(x-cutoff) + switchOver,#lambda x: A*np.exp(-(x - mu)**2 / (2 * sig1**2)),
         lambda x: A/(np.pi)*((sig2)/((x-mu)**2 + sig2**2))
        ]
    )
    

def linGauss(x, mu, sig1, sig2, A, cutoff):
    
    switchOver = A*np.exp(-(cutoff - mu)**2 / (2 * sig2**2))

    return np.piecewise(
        x,
        [x < cutoff - switchOver/sig1,
            (x >= cutoff - switchOver/sig1) & (x < cutoff),
         x >= cutoff],
        [
         0,
         lambda x: sig1*(x-cutoff) + switchOver,#lambda x: A*np.exp(-(x - mu)**2 / (2 * sig1**2)),
         lambda x: A*np.exp(-(x - mu)**2 / (2 * sig2**2))
        ]
    )

def returnXint(mu, sig1, sig2, A, cutoff):
    return cutoff - A*np.exp(-(cutoff - mu)**2 / (2 * sig2**2))/sig1

def doubleGauss(x, mu, sig1, sig2, A, cutoff):
    
    return np.piecewise(
        x,
        [x < mu,
            x >= mu
        ],
        [
         lambda x: A*np.exp(-(x - mu)**2 / (2 * sig1**2)),
         lambda x: A*np.exp(-(x - mu)**2 / (2 * sig2**2))
        ]
    )

def phiFunction_map(x, a, b, c, phi_avg, parIdx, side, max):
    
    #exponent = 1 if parIdx in [0,2] else 1/2
    #denominator = (x - b)/c + 1 if parIdx in [0,2] else np.exp((x - b)/c) + 1
    return np.piecewise(

        x,
        [x < b,
         (x >= b) & (x < max),
         x>max],
        [lambda x: np.full_like(x, phi_avg),
         lambda x: phi_avg + (side)*a**2*(1 - 1/( (x - b)**2/c**2 + 1 if parIdx in [0,2] else np.exp((x - b)/c**2) + 1)),
         lambda x: np.full_like(x, phi_avg)]
    )
def phiFunction_fixed(x, a, c):
    
    denominator = (x - theta_min_fixed)**2/c**2 + 1 if parIdx in [0,2] else np.exp((x - theta_min_fixed)/c**2) + 1
    #exponent = 1 if parIdx in [0,2] else 1/2
    return np.piecewise(

        x,
        [x < theta_min_fixed,
         x >= theta_min_fixed],
        [lambda x: np.full_like(x, phi_avg),
         lambda x:  phi_avg + (side)*a**2*(1 - 1/( denominator + 1))]
    )

def model_phi(B,x):
    exponent = 1 if parIdx in [0] else 1/2
    return np.piecewise(

        x,
        [x < B[1],
         x >= B[1]],
        [lambda x: np.full_like(x, phi_avg),
         lambda x:  phi_avg + (side)*B[0]*(1 - 1/( (x - B[1])**(exponent)/B[2] + 1))]
    )
model = Model(model_phi)
def fitFunction(pIdx):
    if pIdx == 0:
        return doubleGauss
    else:
        return linGauss


In [ ]:
def getHistMean(parIdx, p, sec, hist_0):
    
           
    #bin_contents = np.sum( [hist_0[i].values() for i in range(len(hist_0) )] )  
    bin_contents = np.sum(
    np.stack([hist.values() for hist in hist_0]),
    axis=0
)
    x_centers = np.asarray(hist_0[0].axes[0].centers)
    y_centers = np.asarray(hist_0[0].axes[1].centers)

    values_theta = np.sum(bin_contents, axis=0)
    bound_ind = getPercentile(values_theta, 270, 99)
    max_theta = y_centers[bound_ind]

 
    total_entries = np.sum(bin_contents)

    if total_entries == 0:
        print("The histogram is empty.")
    else:
    # Weighted average for x-axis
    # Reshape bin_contents to match the shape of x_centers for broadcasting
    # or use np.sum with the correct axis.
        sum_x = np.sum(x_centers * np.sum(bin_contents, axis=1))
        mean_x = sum_x / total_entries

        # Weighted average for y-axis
        sum_y = np.sum(y_centers * np.sum(bin_contents, axis=0))
        mean_y = sum_y / total_entries

        return mean_y, mean_x, max_theta
    

In [ ]:
def plot_theta_phi(parIdx, p, sec, ax, hist_0):
    """
    Plot a ROOT TH2 histogram with θ on x-axis and φ on y-axis,
    matching ROOT COLZ style, with percentile cuts.

    Returns:
        min_phi, max_phi, min_theta, max_theta
    """

    # --- Open ROOT file ---
    par = par_list[parIdx]

    hist = hist_0[0]

    # --- Define p-bin range ---
    p_max = 8 if parIdx == 0 else 5
    p_min = 3 if parIdx == 0 else 0
    p_low  = p * (p_max - p_min) / 40. + p_min
    p_high = (p + 1) * (p_max - p_min) / 40. + p_min

    ax.set_title(f"{par}, {p_low:.2f} < p < {p_high:.2f}")

    # --- Extract values and axes ---
    values = hist.values().astype(float)  # shape (theta, phi)
    values[values==0] = np.nan
    theta_edges = hist.axes[1].edges   # θ bins (original y-axis)
    phi_edges   = hist.axes[0].edges    # φ bins (original x-axis)
    theta_centers = hist.axes[1].centers
    phi_centers   = hist.axes[0].centers

    # --- Compute projections for percentile cuts ---
    theta_proj = np.nansum(values, axis=0)  # sum over phi
    phi_proj   = np.nansum(values, axis=1)  # sum over theta

    # --- Safe percentile indexing ---
    max_theta_ind = getPercentile(theta_proj, len(theta_centers), 99)
    min_theta_ind = getPercentile(theta_proj, len(theta_centers), 0.1)
    max_phi_ind   = getPercentile(phi_proj, len(phi_centers), 99)
    min_phi_ind   = getPercentile(phi_proj, len(phi_centers), 0.1)

    max_theta = theta_centers[min(max_theta_ind + 1, len(theta_centers) - 1)]
    min_theta = theta_centers[max(min_theta_ind - 1, 0)]
    max_phi   = phi_centers[min(max_phi_ind + 1, len(phi_centers) - 1)]
    min_phi   = phi_centers[max(min_phi_ind - 1, 0)]

    # --- ROOT-style color scaling (ignore zeros) ---
    zvals = values[values > 0]
    zmin, zmax = np.min(zvals), np.max(zvals)

    # --- Plot: θ on x-axis, φ on y-axis ---

    pcm = ax.pcolormesh(
        theta_edges,   # X-axis
        phi_edges,     # Y-axis
        values,      # transpose to match axes
        shading="flat",#"nearest",
        #vmin=zmin,
        #vmax=zmax,
        cmap="inferno"#"viridis"
    )

    ax.set_xlabel(r"$\theta$")
    ax.set_ylabel(r"$\phi$")
    #plt.colorbar(pcm, ax=ax, label="Counts")

    return min_phi, max_phi, min_theta, max_theta


In [ ]:
t_range = np.linspace( 5, 45, 180)
offsets = [[100,  200, 300,  0, 50],[-100, -70,0, 50, -280, -180]]
params_0 = [[-15, 1, -0.1],[20, -.8, 0.1]  ]

def plot_fixed_theta( parIdx, p, sec, hist):
            
    params=[]
    p_errs=[]
    yVals=[]
                    
    values = sum([np.array(hist[i].values()) for i in range(len(hist))])
    xEdges = hist[0].axes[0].centers
    xExtrema = np.where(np.all(values > 0, axis = 0))[0]
    yEdges = hist[0].axes[1].centers
    yExtrema = np.where(np.all(values > 0, axis = 1))[0]

    
    for theta in range(len(yEdges)):
                                 
        if np.sum(values[:,theta])<=100:
            continue

        pv = np.where(values[:,theta]>0.65*np.max(values[:,theta]))
        gtz = np.where(values[:,theta]>0)
        
        p0 = [xEdges[gtz[0][0]], 0.5*(xEdges[gtz[0][0]]+xEdges[pv[0][0]]), 0.5*(xEdges[pv[0][-1]]+xEdges[pv[0][-1]]),xEdges[gtz[0][-1]], 0.75*np.max(values[:,theta])]

        try:
            popt, pcov = curve_fit(trapezoid, xEdges, values[:,theta], p0)
        except RuntimeError as e:
            #print("Failed phi fit")
            continue
        if popt[0] > popt[1] or popt[1] > popt[2] or popt[2] > popt[3]:
            continue
        #if popt[1] > (popt[0]+popt[3])/2. or popt[2] < (popt[0]+popt[3])/2.:
        #    continue

        perr = np.sqrt(np.diag(pcov))
        
        #if popt[0] <xEdges[gtz[0][0]] or popt[3] > xEdges[gtz[0][-1]]:
        #    continue
        
        params.append([popt[i] for i in range(4)])       
        p_errs.append([perr[i] for i in range(4)])
        yVals.append(yEdges[theta])
        
        #fig = plt.figure()
        #ax=fig.add_subplot(111)
        #    ax.plot(yEdges, values[phi])
        #ax.set_title(f"phi = {-250 + phi}")
        #plt.plot(xEdges, values[:,theta])
        #plt.plot(xEdges, trapezoid(xEdges, *popt))
        #ax.set_xlim(popt[0]-5, popt[3]+5)

    return np.asarray(params), np.asarray(p_errs), np.asarray(yVals)

In [ ]:
phi_range = np.linspace(0,45, 500)
cut_0 =[25, 10, 24]
def plot_fixed_phi( parIdx, p, sec, hist):
    
    par = par_list[parIdx]
            
    
    low = []
    low_err = []
    xVals=[]
                    
    values = sum([np.array(hist[i].values()) for i in range(len(hist))])
    xEdges = hist[0].axes[0].centers
    yEdges = hist[0].axes[1].centers

    for phi in range(len(xEdges)):
        if np.sum(values[phi])<=100:
            continue
    
        max_theta_idx = np.where(values[phi]>.9*np.max(values[phi]))
        max_theta_idx_cut = np.where(values[phi]>0.8*np.max(values[phi]))
        p0 = [yEdges[max_theta_idx[0][0]],1, 5, np.max(values[phi]), cut_0[parIdx]]
        
        gtz = np.where(values[phi,:]>0)[0]
        try:
            popt, pcov = curve_fit(doubleGauss, yEdges, values[phi], p0=p0)
        except RuntimeError as e:
            print("Theta fit failed")
            continue
        if parIdx > 0:
            std_dev = np.sqrt(np.average((yEdges - np.average(yEdges, weights=values[phi]))**2, weights=values[phi]))
            
            p0 = [yEdges[max_theta_idx[0][0]],200, std_dev, np.max(values[phi]), (yEdges[max_theta_idx_cut[0][0]])]
            try:
                popt, pcov = curve_fit(linGauss, yEdges, values[phi], p0=p0)
            except RuntimeError as e:
                #print("Theta Fit Failed")
                continue
        #if parIdx > 0 and popt[4] < yEdges[gtz[0]]-5:
        #    continue
        #if parIdx == 0 and popt[0] < yEdges[gtz[0]]:
        #    continue
        #if sec == 4:
        #   fig = plt.figure()
        #    ax=fig.add_subplot(111)
        #    ax.plot(yEdges, values[phi])
        #ax.set_title(f"phi = {-250 + phi}")
            #plt.plot(yEdges, values[phi])
            #plt.plot(yEdges, linGauss(yEdges, *popt))
        #ax.set_xlim(popt[0]-5, popt[3]+5)
        perr = np.sqrt(np.diag(pcov))
        if parIdx > 0:
            #low.append(returnXint(*popt))
            low.append(popt[4])
            low_err.append(perr[4])
            #low_err.append(popt[4] - returnXint(*popt))
        else:
            low.append(popt[0])
            low_err.append(perr[1])
        xVals.append(xEdges[phi])
    return np.asarray(low), np.asarray(low_err), np.asarray(xVals)

In [ ]:
def applyMask(pt_array, bin_array, factor):
    slope = (pt_array[1:] - pt_array[:-1])/(bin_array[1:] - bin_array[:-1])
    #good_intervals = slope < 1
    #intervals_sign = slope > 0 
    mask_upper = np.insert(slope<10, 0, False)

    fig = plt.figure()
    ax=fig.add_subplot(111)
    ax.plot(bin_array[1:], slope)
    
    #mask_points = np.ones(len(pt_array), dtype=bool)
    #mask_points[:-1] &= good_intervals
    #mask_points[:-1] &= intervals_sign
    #mask_points[1:] &= good_intervals

    return mask_upper

In [ ]:
def doAnalysis(parIdx):

    i=parIdx
    side = 1
    phi_avg =0
    theta_min_fixed = 0

    def phiFunction(x, a, b, c):
        
        #exponent = 1 if parIdx in [0,1,2] else 1/2
        return np.piecewise(

            x,
            [x < b,
            x >= b],
            [lambda x: np.full_like(x, phi_avg),
            lambda x:  phi_avg + (side)*a**2*(1 - 1/( ((x - b)/c**2 + 1 if parIdx in [0,2] else np.exp((x - b)/c**2) + 1) ))]
        )
    def phiFunction_fixed(x, a, c):
       
        #exponent = 1 if parIdx in [0,1,2] else 1/2
        return np.piecewise(

            x,
            [x < theta_min_fixed,
            x >= theta_min_fixed],
            [lambda x: np.full_like(x, phi_avg),
            lambda x:  phi_avg + (side)*a**2*(1 - 1/( ((x - theta_min_fixed)/c**2 + 1 if parIdx in [0,2] else np.exp((x - theta_min_fixed)/c**2) + 1)))]
        )

    params_l1 = [[] for i in range(6)]
    params_l2 = [[] for i in range(6)]
    params_l3 = [[] for i in range(6)]
    params_u1 = [[] for i in range(6)]
    params_u2 = [[] for i in range(6)]

    errs_l1 = [[] for i in range(6)]
    errs_l2 = [[] for i in range(6)]
    errs_l3 = [[] for i in range(6)]
    errs_u1 = [[] for i in range(6)]
    errs_u2 = [[] for i in range(6)]
   


    p_low = 0 if i == 0 else 6 #6
    p_high = 32 if i == 0 else 38
    Range = range(p_low, p_high) if i==0 else reversed(range(p_low, p_high))
    p0 = [8, 8, 1.5]
    for p in Range:#reversed(range(p_low,7)):
       
        
        fig, ax = plt.subplots(1, 2, figsize=(10,12), layout='constrained')
      
        # Initialize global extrema
        phi_min_g   =  [999, 999]
        phi_max_g   = [-999, -999]
        theta_min_g =  999
        theta_max_g = -999
        for sector in [0,1,2,3,4,5]:
            pan = 0 if sector in [1,2,3] else 1

            par = par_list[parIdx]

            file_0 = [uproot.open(tfile_hist[i]) for i in range(len(tfile_hist))]
            hist_0 = [file_0[i][hBase + f'_{sector}_bin_{p}_{par};1'] for i in range(len(file_0))]
            
            

            hist_0 = [hist.to_boost() for hist in hist_0]

            if( p > 30 ):# and p%2 == 0):
                #for file in file_0:
                #    hist_0.append(file[hBase + f'_{sector}_bin_{p-1}_{par};1'] )
                #hist_0.append(file_0[hBase + f'_{sector}_bin_{p-2}_{par};1'])
                #hist_0.append(file_0[hBase + f'_{sector}_bin_{p-3}_{par};1'])   
                hist_0 = [hist[::bh.rebin(1), ::bh.rebin(2)] for hist in hist_0]    
            #elif( p > 30 and p%2 != 0):
            #    continue
     

            low, low_err, thetaVals = plot_fixed_phi( i, p, sector, hist_0) #get "gaussian" fits
            params, perr, phiVals = plot_fixed_theta(i, p, sector, hist_0) #get trapezoidal fits
            
            upper_pts = params[:,2]
            lower_pts = params[:,1]

            up_err = perr[:,2] 
            bot_err = perr[:,1]
            
            params=np.asarray(params)
            
            
            theta_mean, phi_avg, max_theta = getHistMean(i, p, sector, hist_0)
            phi_min, phi_max, theta_min, theta_max = plot_theta_phi(i, p, sector, ax[pan], hist_0)

            # Accumulate loosest limits
            phi_min_g[pan]   = min(phi_min_g[pan],   phi_min)
            phi_max_g[pan]   = max(phi_max_g[pan],   phi_max)
            theta_min_g = min(theta_min_g, theta_min)
            theta_max_g = max(theta_max_g, theta_max)
            
            #lower_pts = savgol_filter(upper_pts, window_length=10, polyorder=3)        

            ax[pan].plot( phiVals, upper_pts , color='cyan',ms=2, marker='o', linestyle=' ') #bottom line
            ax[pan].plot( phiVals, lower_pts, color='cyan', marker='o', linestyle=' ',ms=2)
            ax[pan].plot( low, thetaVals, color='cyan', marker='o', linestyle=' ',ms=2)

            #ax[pan].plot( low, thetaVals, marker='+', ms = 4, linestyle=' ', color='red')
        
            mask_upper = applyMask(upper_pts, phiVals, 1)
            upper_pts = np.where(mask_upper, np.nan, upper_pts)
            upper_pts = upper_pts[~np.isnan(upper_pts)]
            yVals_upper = np.where(mask_upper, np.nan, phiVals)
            yVals_upper = yVals_upper[~np.isnan(yVals_upper)]
     
            mask_lower = applyMask(lower_pts, phiVals, 1)
            lower_pts = np.where(mask_lower, np.nan, lower_pts)
            lower_pts = lower_pts[~np.isnan(lower_pts)]
            yVals_lower = np.where(mask_lower, np.nan, phiVals)
            yVals_lower = yVals_lower[~np.isnan(yVals_lower)]
            
            mask_left = applyMask(thetaVals, low, 1)

            #print(phiVals)
            #print(upper_pts)

            ax[pan].plot( yVals_upper, upper_pts , color='w',ms=2, marker='o', linestyle=' ') #bottom line
            ax[pan].plot( yVals_lower, lower_pts, color='w', marker='o', linestyle=' ',ms=2)
            ax[pan].plot( low[mask_left], thetaVals[mask_left], color='lime', marker='o', linestyle=' ',ms=2)


            mom = 3*(parIdx < 1) + p*1/8

            low_mask = (thetaVals < phi_avg)#&(xVals < np.min(params_2))
            high_mask = (thetaVals>= phi_avg)#&(xVals >= np.min(params_1))
            
            low_vals = low[low_mask]
            high_vals = low[high_mask]
            
            xVals_low = thetaVals[low_mask]
            xVals_high = thetaVals[high_mask]
        
            
            
            #ax[pan].plot( low_vals, xVals_low, marker='.', ms = 4, linestyle=' ', color='white')
            #ax[pan].plot( high_vals, xVals_high, marker='.', ms = 4, linestyle=' ', color='white')



            side = -1


        
            #ax[pan].plot( phi_range, phiFunction(phi_range, *output_up.beta), color='red')
            popt_up, pcov_up = curve_fit(phiFunction, low_vals, xVals_low, nan_policy='omit', p0 = p0, maxfev=15000000)
            
            perr_up = np.sqrt(np.diag(pcov_up))
            p0 = popt_up#output_up.beta#popt_up
            
            side=1
            popt_low, pcov_low = curve_fit(phiFunction, high_vals, xVals_high, nan_policy='omit',p0 = p0, maxfev=15000000)
            perr_low = np.sqrt(np.diag(pcov_low))
           
            p0 = popt_low
            theta_min_fixed = (popt_low[1]+popt_up[1])/2
            theta_min_err = 0.5*np.sqrt((perr_up[1])**2 + (perr_low[1])**1)

            p0_2 = [(popt_up[j]+popt_low[j])/2 for j in [0,2]]
            
            side = -1
            popt_up, pcov_up = curve_fit(phiFunction_fixed, low_vals, xVals_low, nan_policy='omit', p0 = p0_2, maxfev=15000000)
            side=1
            popt_low, pcov_up = curve_fit(phiFunction_fixed, high_vals, xVals_high, nan_policy='omit', p0 =p0_2, maxfev=15000000)
            

            side=-1
            ax[pan].plot( phi_range, phiFunction_fixed(phi_range, *popt_up),color='red')
            side=1
            ax[pan].plot( phi_range, phiFunction_fixed(phi_range, *popt_low),color='red')

        

            perr_low = np.sqrt(np.diag(pcov_low))
            perr_up = np.sqrt(np.diag(pcov_up))

            insIdx = 0 if parIdx > 0 else np.size(params_l1[sector])

            params_l1[sector].insert(insIdx, popt_low[0])
            params_l2[sector].insert(insIdx,popt_low[1])
            params_l3[sector].insert(insIdx,theta_min_fixed)

            params_u1[sector].insert(insIdx,popt_up[0])
            params_u2[sector].insert(insIdx,popt_up[1])
            errs_l1[sector].insert(insIdx,perr_low[0])
            errs_l2[sector].insert(insIdx,perr_low[1])
            errs_l3[sector].insert(insIdx,theta_min_err)


            errs_u1[sector].insert(insIdx,perr_up[0])
            errs_u2[sector].insert(insIdx,perr_up[1])
        ax[0].set_xlim(theta_min_g, theta_max_g)
        ax[1].set_xlim(theta_min_g, theta_max_g)

        ax[0].set_ylim(phi_min_g[0], phi_max_g[0])
        ax[1].set_ylim(phi_min_g[1], phi_max_g[1])
    return np.abs(params_l1), np.abs(params_l2), params_l3,  np.abs(params_u1), np.abs(params_u2),errs_l1, errs_l2, errs_l3, errs_u1, errs_u2


In [ ]:
params_pim_l1, params_pim_l2, params_pim_l3,  params_pim_u1, params_pim_u2,errs_pim_l1, errs_pim_l2, errs_pim_l3, errs_pim_u1, errs_pim_u2 = doAnalysis(2)

In [ ]:
params_pip_l1, params_pip_l2, params_pip_l3,  params_pip_u1, params_pip_u2,errs_pip_l1, errs_pip_l2, errs_pip_l3, errs_pip_u1, errs_pip_u2 = doAnalysis(1)

In [ ]:
params_l1, params_l2, params_l3,  params_u1, params_u2,errs_l1, errs_l2, errs_l3, errs_u1, errs_u2 = doAnalysis(0)

In [ ]:
#electron and pip functions
def func_theta_min_el(x, p0, p1, p2, p3, p4, p5):
    return p0+ p1/x**2 +p2*x+ p3/x + p4*np.exp(p5*x) 
def func_el_a(x, p0, p1, p2, p3):
    return p0 + p1*np.exp(p2*(x-p3))
def func_el_b(x, p0, p1, p2, p3):
    return p0 + p1*x*np.exp(p2*(x-p3)**2)

def func_pim_a(x, p0, p1, p2, p3):
    return p0 - p1*np.arctan(p2*(x-p3))
def func_pim_b(x, p0, p1, p2, p3):
    return p0 + p1*x**2*np.exp(p2*(x-p3)**2)
    #return func_el_b(x, p0, p1, p2, p3)

def returnMap(x, p, theta_params, params, sector, phi_avg, parIdx, max):
    minimum_theta = func_theta_min_el(p, *theta_params[sector])   
    param_a_l, param_a_u, param_b_l, param_b_u = 0,0,0,0 
    
    
    param_a_l = func_el_a(p, *params[0][sector]) if parIdx < 2 else func_el_b(p, *params[0][sector])
    param_a_u =  func_el_a(p, *params[1][sector]) if parIdx < 2 else func_el_b(p, *params[1][sector])

    param_b_l =  func_el_b(p, *params[2][sector]) if parIdx < 2 else func_el_b(p, *params[2][sector])
    param_b_u =  func_el_b(p, *params[3][sector]) if parIdx < 2 else func_el_b(p, *params[3][sector])


    

  
    l_map = phiFunction_map(x, param_a_l,  minimum_theta, param_b_l,phi_avg, parIdx, -1, max)
    
    u_map = phiFunction_map(x, param_a_u,  minimum_theta, param_b_u, phi_avg, parIdx, 1, max) 

    return l_map, u_map


In [ ]:
pi_range = np.linspace(0 + (5)/40, 5 - 5/40, 40)[7:38]

pim_theta_min_params =[]
pim_params = [[] for i in range(4)]

fig, ax = plt.subplots(2, 3, figsize=(10,12), layout='constrained')

p0=[
[22.69737102,  1.43862297, 53.72860297 , 4.76626883],
[ 25.22936743,  -0.58420767 ,198.03492013 ,  3],
[ 25.22936743,  -0.58420767 ,198.03492013 ,  1.34938381],
[26.84862117, -1.5683479,   2.46165834 , 1.95532219],
[26.15585521, -0.64291964, 18.286519,    1.96141262],
[ 25.22936743,  -0.58420767 ,198.03492013 ,  1.34938381]
]

p0=[
    [ 1.38734212e+01 , 1.58092717e+46 ,-3.75765050e-04, -5.25489581e+02],
    [ 1.38734212e+01 , 1.58092717e+46 ,-3.75765050e-04, -5.25489581e+02],
    [23.42833254 , 2.27471791, -0.17703057,  2.3239606 ],
    [ 1.38734212e+01 , 1.58092717e+46 ,-3.75765050e-04, -5.25489581e+02],
    [ 1.38734212e+01 , 1.58092717e+46 ,-3.75765050e-04, -5.25489581e+02],
    [26.1495786 , -2.37813622 , 1.41841281 , 1.35933975]
]

params_pim_l1 =  np.asarray(params_pim_l1)
params_pim_u1 =  np.asarray(params_pim_u1)

errs_pim_l1= np.asarray(errs_pim_l1)
errs_pim_u1= np.asarray(errs_pim_u1)

for i in range(0,6):
    extra_mask_u = (params_pim_u1[i][1:32]<100)&(errs_pim_u1[i][1:32]<100)
    extra_mask_l=  (params_pim_l1[i][1:32]<100)&(errs_pim_u1[i][1:32]<100)    

    ax[i%2, math.floor(i/2)].errorbar(pi_range[extra_mask_l], params_pim_l1[i][1:32][extra_mask_l], errs_pim_l1[i][1:32][extra_mask_l], marker = '.', linestyle='-', color='blue')
    popt_a, pcov_a = curve_fit(func_el_b, pi_range[extra_mask_l], 
                               params_pim_l1[i][1:32][extra_mask_l], 
                               sigma=errs_pim_l1[i][1:32][extra_mask_l], 
                               maxfev=1000000, 
                              p0=[25.24037743  ,1.61955564, -0.20310906  ,4.17053295])#p0[i])
    ax[i%2, math.floor(i/2)].plot(np.linspace(0.75, 5, 100), func_el_b(np.linspace(0.75, 5, 100), *popt_a), color='navy')
    pim_params[0].append(popt_a)

    ax[i%2, math.floor(i/2)].errorbar(pi_range[extra_mask_u], params_pim_u1[i][1:32][extra_mask_u], errs_pim_u1[i][1:32][extra_mask_u], marker = '.', linestyle='-', color='red')
    popt_a, pcov_a = curve_fit(func_el_b, pi_range[extra_mask_u],
                                params_pim_u1[i][1:32][extra_mask_u], 
                                sigma=errs_pim_u1[i][1:32][extra_mask_u], 
                                maxfev=1500000000, 
                                p0=[3.86135093e+00 , 1.12485333e+49, -3.42980638e-04 ,-5.73038843e+02])
    ax[i%2, math.floor(i/2)].plot(np.linspace(0.5, 5, 100), func_el_b(np.linspace(0.5, 5, 100), *popt_a), color='maroon')
    pim_params[1].append(popt_a)
    print(popt_a)
fig, ax = plt.subplots(2, 3, figsize=(10,12), layout='constrained')
params_pim_l2 =  np.asarray(params_pim_l2)
params_pim_u2 =  np.asarray(params_pim_u2)

errs_pim_l2= np.asarray(errs_pim_l2)
errs_pim_u2= np.asarray(errs_pim_u2)

p0=[
    [0.02451813 , 1.16576816, -0.08641484 , 2.0009107 ],
    [0.02451813 , 1.16576816, -0.08641484 , 2.0009107 ],
    [ 1.46149857 , 0.09839648, -0.37850987,  3.03900145],#[-0.029102886 , 1.08589699,  -0.84423735 , -.514579099 ],#[-0.29102886 , 0.38589699,  0.04423735 , 3.64579099 ],
    [0.02451813 , 1.16576816, -0.08641484 , 2.0009107 ],
    [0.02451813 , 1.16576816, -0.18641484 , -0.10009107 ],
    [0.02451813 , 1.16576816, -0.08641484 , 2.0009107 ]
]

for i in range(0,6):
    extra_mask_u = (params_pim_u2[i][1:32]<1000)&(errs_pim_u2[i][1:32]<1000)&(params_pim_u2[i][1:32]>0)
    extra_mask_l=  (params_pim_l2[i][1:32]<1000)&(errs_pim_l2[i][1:32]<1000)&(params_pim_l2[i][1:32]>0)
    ax[i%2, math.floor(i/2)].errorbar(pi_range[extra_mask_l], params_pim_l2[i][1:32][extra_mask_l], errs_pim_l2[i][1:32][extra_mask_l], marker = '.', linestyle='-', color='blue')
    popt_b, pcov_b = curve_fit(func_el_b, pi_range[extra_mask_l], 
                               params_pim_l2[i][1:32][extra_mask_l], 
                               sigma=errs_pim_l2[i][1:32][extra_mask_l], 
                               maxfev=1500000,
                               p0=[-0.02451813 , 1.16576816, -0.08641484 , 2.0009107 ])
    ax[i%2, math.floor(i/2)].plot(np.linspace(0.5, 5, 100), func_el_b(np.linspace(0.5, 5, 100), *popt_b), color='navy')
    pim_params[2].append(popt_b)
  
    ax[i%2, math.floor(i/2)].errorbar(pi_range[extra_mask_u], params_pim_u2[i][1:32][extra_mask_u], errs_pim_u2[i][1:32][extra_mask_u], marker = '.', linestyle='-', color='red')
    popt_b, pcov_b = curve_fit(func_el_b, pi_range[extra_mask_u], 
                               params_pim_u2[i][1:32][extra_mask_u], 
                               sigma=errs_pim_u2[i][1:32][extra_mask_u],
                               maxfev=1200000,
                               p0=p0[i])
    ax[i%2, math.floor(i/2)].plot(np.linspace(0.75, 5, 100), func_el_b(np.linspace(0.75, 5, 100), *popt_b), color='maroon')
    #ax[i%2, math.floor(i/2)].set_ylim(0,4)
    pim_params[3].append(popt_b)
    #print(popt_b)

  

fig, ax = plt.subplots(2, 3, figsize=(10,12), layout='constrained')

for i in range(0,6):
    
    ax[i%2, math.floor(i/2)].errorbar(pi_range, params_pim_l3[i][1:32], errs_pim_l3[i][1:32], marker = '.', linestyle='-', color='blue')

    popt_t, pcov_t = curve_fit(func_theta_min_el, pi_range, params_pim_l3[i][1:32], sigma=errs_pim_l3[i][1:32], maxfev=2500000, p0=[-2.90707390e+00, -2.00636024e+01 , 1.47550799e+00 , 4.10308689e+01,
 -3.01837933e-02 , 8.87328903e-01])
    ax[i%2, math.floor(i/2)].plot(np.linspace(0.5, 5, 100), func_theta_min_el(np.linspace(0.5, 5, 100), *popt_t), color='navy')
    pim_theta_min_params.append(popt_t)

In [ ]:
pi_range = np.linspace(0 + (5)/40, 5 - 5/40, 40)[9:40]

print(len(pi_range))
print(len(params_pip_l1[0][1:34]))
fig, ax = plt.subplots(2, 3, figsize=(10,12), layout='constrained')
pip_theta_min_params =[]
pip_params = [[] for i in range(4)]


p0 = [
    [ 4.89202304, -0.03593582,  0.53783636,  0.2211076 ],
    [ 37.3619118,  -5.21415498 , 0.28460749,  3.36062314],
    [32.75239255 ,-5.1851922,  0.59822379 , 1.05829656],
    [34.02641177, -1.00576906,  0.62178092,  1.3605207  ],
    [ 32.75239255 ,-5.1851922,  0.59822379 , 1.05829656 ],
    [32.75239255 ,-5.1851922,  0.59822379 , 1.05829656]]

p0_2 = [
[ 4.7878744,  -0.03543518,  1.04671056,  2.93439632],
[ 4.7878744,  -0.03543518,  1.04671056,  2.93439632],
[ 4.7878744,  -0.03543518,  1.04671056,  2.93439632],
[ 4.7878744,  -0.03543518,  1.04671056,  2.93439632],
[ 4.7878744,  -0.03543518,  1.04671056,  2.93439632],
[ 4.7878744,  -0.03543518,  1.04671056,  2.93439632]]


for i in range(0,6):
    
    ax[i%2, math.floor(i/2)].errorbar(pi_range, params_pip_l1[i][1:34],errs_pip_l1[i][1:34], marker = '.', linestyle='-', color='blue')
    popt_a, pcov_a = curve_fit(func_el_a, pi_range, params_pip_l1[i][1:34],sigma = errs_pip_l1[i][1:34], maxfev=150000000, p0=p0[i])
    ax[i%2, math.floor(i/2)].plot(np.linspace(0, 5, 100), func_el_a(np.linspace(0, 5, 100), *popt_a), color='navy')
    pip_params[0].append(popt_a)
    print(popt_a)

    ax[i%2, math.floor(i/2)].errorbar(pi_range, params_pip_u1[i][1:34], errs_pip_u1[i][1:34], marker = '.', linestyle='-', color='red')
    popt_a, pcov_a = curve_fit(func_el_a, pi_range, params_pip_u1[i][1:34],sigma = errs_pip_u1[i][1:34], maxfev=150000000, p0=p0_2[i])#[ 25.10526297, -1.48492098 ,  1.05417794 ,  6.1916069 ])
    ax[i%2, math.floor(i/2)].plot(np.linspace(0, 5, 100), func_el_a(np.linspace(0, 5, 100), *popt_a), color='maroon')
    pip_params[1].append(popt_a)
    print(popt_a)

    #ax[i%2, math.floor(i/2)].set_ylim(0, 50)

fig, ax = plt.subplots(2, 3, figsize=(10,12), layout='constrained')

p0=[
[ 2.71738185, -0.18481952,  0.13858209,  3.88785433],
[2.71738185, -0.18481952,  0.13858209,  3.8878543],
[2.71738185, -0.18481952,  0.13858209,  3.8878543],
[1.34319222, 0.06033968, 0.4212403, 3.71159105],
[1.34319222, 0.06033968, 0.4212403, 3.71159105],
[0.99058498, 0.14595902, 0.14182083, 5.11437383]
]

for i in range(0,6):
    
    ax[i%2, math.floor(i/2)].errorbar(pi_range, params_pip_l2[i][1:34], errs_pip_l2[i][1:34], marker = '.', linestyle='-', color='blue')
    popt_b, pcov_b = curve_fit(func_el_b, pi_range, params_pip_l2[i][1:34],  maxfev=15000000,p0 =[ 3.3106001 , -0.32518597 , 0.07038956 , 4.99044399])#[3.20859722,0.01368572, 0.17229598, 6.30687601])
    ax[i%2, math.floor(i/2)].plot(np.linspace(0.5, 5, 100), func_el_b(np.linspace(0.5, 5, 100), *popt_b), color='navy')
    pip_params[2].append(popt_b)

    ax[i%2, math.floor(i/2)].errorbar(pi_range, params_pip_u2[i][1:34], errs_pip_u2[i][1:34], marker = '.', linestyle='-', color='red')
    popt_b, pcov_b = curve_fit(func_el_b, pi_range, params_pip_u2[i][1:34], p0 =[1.03281288, 0.1153517 , 0.11677437, 5.84449444],maxfev=150000000)
    #print(popt_b)
    ax[i%2, math.floor(i/2)].plot(np.linspace(0.75, 5, 100), func_el_b(np.linspace(0.75, 5, 100), *popt_b), color='maroon')
    pip_params[3].append(popt_b)
    #print(popt_b)



fig, ax = plt.subplots(2, 3, figsize=(10,12), layout='constrained')

for i in range(0,6):
    
    ax[i%2, math.floor(i/2)].errorbar(pi_range, params_pip_l3[i][1:34], errs_pip_l3[i][1:34], marker = '.', linestyle='-', color='blue')

    popt_t, pcov_t = curve_fit(func_theta_min_el, pi_range, params_pip_l3[i][1:34], maxfev=15000000,p0=[ 3.93265139e+00, -6.35440228e-01,  1.11616750e+00 , 3.38232829e+00,
  1.13480678e-03, -3.57516466e-01])
    #print(popt_t)
    ax[i%2, math.floor(i/2)].plot(np.linspace(0.5, 5, 100), func_theta_min_el(np.linspace(0.5, 5, 100), *popt_t), color='navy')
    pip_theta_min_params.append(popt_t)

   

In [ ]:
el_theta_min_params =[]
e_params = [[] for i in range(4)]
e_range = np.linspace( 3 + (8-3)/80, 7 - 5/80, 32)
fig, ax = plt.subplots(2, 3, figsize=(10,12), layout='constrained')

p0=[[5.34724932e+00, 8.92968683e-05, 2.05384568e+00, 2.38442634e+00],
    [5.34724932e+00, 8.92968683e-05, 2.05384568e+00, 2.38442634e+00],
    [5.34724932e+00, 8.92968683e-05, 2.05384568e+00, 2.38442634e+00],
    [5.34724932e+00, 8.92968683e-05, 2.05384568e+00, 2.38442634e+00],
    [5.34724932e+00, 80.92968683e-05, 2.05384568e+00, 2.38442634e+00],
    [5.34724932e+00, 100.92968683e-05, 2.05384568e+00, 2.38442634e+00],
    ]

p0_2=[[5.01999708 ,0.0099818  ,0.51779686 ,0.07449413],
[5.01999708, 0.00000099818,  0.51779686, 0.00007449413],
[5.01999708 ,0.0099818  ,0.51779686 ,0.07449413],

[5.01999708 ,0.0099818  ,0.51779686 ,0.07449413],

[5.01999708 ,0.1099818  ,0.51779686 ,0.007449413],

[5.01999708 ,0.0099818  ,0.51779686 ,0.007449413]]

for i in range(0,6):
    ax[i%2, math.floor(i/2)].errorbar(e_range, params_l1[i][0:32], errs_l1[i][0:32], marker = '.', linestyle='-', color='blue')
    popt_a, pcov_a = curve_fit(func_el_a, e_range, params_l1[i][0:32], sigma=errs_l1[i][0:32], p0=p0[i], maxfev=150000000)
    ax[i%2, math.floor(i/2)].plot(np.linspace(3, 7, 100), func_el_a(np.linspace(3, 7, 100), *popt_a), color='navy')

    e_params[0].append(popt_a)

    ax[i%2, math.floor(i/2)].errorbar(e_range, params_u1[i][0:32], errs_u1[i][0:32], marker = '.', linestyle='-', color='red')
    popt_a, pcov_a = curve_fit(func_el_a, e_range, params_u1[i][0:32], sigma=errs_u1[i][0:32], p0=p0_2[i], maxfev=100000000)
    ax[i%2, math.floor(i/2)].plot(np.linspace(3, 7, 100), func_el_a(np.linspace(3, 7, 100), *popt_a), color='maroon')
    e_params[1].append(popt_a)
    print(popt_a)
   
##########################################
##########################################
##########################################


fig, ax = plt.subplots(2, 3, figsize=(10,12), layout='constrained')

p0=[
[3.67070278, 0.06004964, 0.18021776, 3.2877174 ],
[-2.44576369, 1.05193331, 0.0444404,  6.42218139],
[-2.44576369, 1.05193331, 0.0444404,  6.42218139],
[3.81586986, 0.04470739, 0.28428386, 3.69236709],
[3.85189111, 0.03299414, 0.24744023, 3.3726926 ],
[3.85189111, 0.03299414, 0.24744023, 3.3726926 ]]

for i in range(0,6):
    
    ax[i%2, math.floor(i/2)].errorbar(e_range[0:-2], params_l2[i][0:30], errs_l2[i][0:30], marker = '.', linestyle='-', color = 'blue')
    popt_b, pcov_b = curve_fit(func_el_b, e_range[0:-2], params_l2[i][0:30], sigma=errs_l2[i][0:30], p0 =p0[i][0:30], maxfev=15000000)
    ax[i%2, math.floor(i/2)].plot(np.linspace(3, 7, 100), func_el_b(np.linspace(3, 7, 100), *popt_b), color= 'navy')
    e_params[2].append(popt_b)

    ax[i%2, math.floor(i/2)].errorbar(e_range, params_u2[i][0:32], errs_u2[i][0:32], marker = '.', linestyle='-', color = 'red')
    popt_b, pcov_b = curve_fit(func_el_b, e_range, params_u2[i][0:32], sigma=errs_u2[i][0:32], p0 =[-2.44576369, 1.05193331, 0.0444404,  6.42218139], maxfev=150000000)
    ax[i%2, math.floor(i/2)].plot(np.linspace(3, 7, 100), func_el_b(np.linspace(3, 7, 100), *popt_b), color= 'maroon')
    e_params[3].append(popt_b)
       
##########################################
##########################################
##########################################

fig, ax = plt.subplots(2, 3, figsize=(10,12), layout='constrained')
p0 = [[5.80421027e+00, -1.44454221e+01 , 6.22691652e-04 , 2.36592337e+01,
  3.28999322e-06 , 1.36728664e+00],
      [5.80421027e+00, -1.44454221e+01 , 6.22691652e-04 , 2.36592337e+01,
  3.28999322e-06 , 1.36728664e+00],
[  -6.25527368, -101.87371682,    1.44266018 ,  82.69214178,   -0.13908512, 0.5716702 ],
[ 8.05825836,  8.41737744,  1.07303332 , 9.90951454, -0.77564392,  0.16463166],
[5.80421027e+00, -1.44454221e+01 , 6.22691652e-04 , 2.36592337e+01,
  3.28999322e-06 , 1.36728664e+00],
[5.80421027e+00, -1.44454221e+01 , 6.22691652e-04 , 2.36592337e+01,
  3.28999322e-06 , 1.36728664e+00]]


for i in range(0,6):
    ax[i%2, math.floor(i/2)].errorbar(e_range, params_l3[i][0:32], errs_l3[i][0:32], marker = '.', linestyle='-', color = 'blue')
    popt_t, pcov_t = curve_fit(func_theta_min_el, e_range, params_l3[i][0:32], sigma=errs_l3[i][0:32], p0=p0[i][0:32], maxfev=1500000)
    ax[i%2, math.floor(i/2)].plot(np.linspace(3, 7, 100), func_theta_min_el(np.linspace(3, 7, 100), *popt_t), color= 'navy')

    el_theta_min_params.append(popt_t)
  


In [ ]:
parIdx = [0,1,2]

max_par = [[[] for sec in range(6)] for par in range(3)]
mean_par = [[[] for sec in range(6)] for par in range(3)]



fig, ax = plt.subplots(2, 3, figsize=(10,12), layout='constrained')
for i in parIdx:
    ind_min = 7 if i==1 else 7

    p_pi = np.linspace(0,5,40)[ind_min:40]
    p_e = np.linspace(3, 8,40)[0:32]

    side = 1
    phi_avg =0
    theta_min_fixed = 0

    p_low = 0 if i == 0 else ind_min
    p_high = 32 if i == 0 else 40
    Range = range(p_low, p_high)


    for sector in [0,1,2,3,4,5]:
        max_array = []
        mean_array = []
        for p in Range:

            y_mean, x_mean, max_theta = getHistMean(i, p, sector)        

            max_array.append(max_theta)
            mean_array.append(x_mean)
            
        ax[sector%2, math.floor(sector/2)].plot(p_e if i == 0 else p_pi, mean_array, marker='.', linestyle=' ')
        #print(p_e)
        #print(mean_array)
        popt_theta, _ = curve_fit(poly3, p_e if i == 0 else p_pi, max_array)
        popt_mean, _ = curve_fit(expo, p_e if i == 0 else p_pi, mean_array, p0=[20-40*(i==1), 0,0.3, 2], maxfev=100000000)

        ax[sector%2, math.floor(sector/2)].plot(np.linspace(0, 5+ (i<1)*3, 100), expo(np.linspace(0, 5 + (i<1)*3, 100), *popt_mean), marker = ' ', linestyle='-')
        #if i ==0:
        #    print(popt_theta)
        max_par[i][sector].append(popt_theta)
        mean_par[i][sector].append(popt_mean)

In [ ]:
def plotMaps(parIdx):
    i = parIdx
    side = 1
    phi_avg =0
    theta_min_fixed = 0

    p_min = 0 if i == 0 else 6
    p_max = 32 if i == 0 else 40
    Range = range(p_min, p_max) if i==0 else reversed(range(p_min, p_max))
    for p in Range:
        
        fig, ax = plt.subplots(1, 2, figsize=(10,12), layout='constrained')
        for sector in [0,1,2,3,4,5]:
            pan = 0 if sector in [1,2,3] else 1
            
            
            plot_fit(parIdx, p, sector, ax[pan])
            y_mean, x_mean, max_theta = getHistMean(parIdx, p, sector)
            low_val = 10#np.min(low) if np.min(low) > 0 else 10
        
            

            ax[0].set_xlim(low_val-5, 35)
            ax[1].set_xlim(low_val-5, 35)

            ax[0].set_ylim(-10, 225)
            ax[1].set_ylim(-175,35)

            p_low = 3*(parIdx == 0) + p*(5/40)
            p_mid = 3*(parIdx == 0) + (p+.5)*(5/40)
            p_high = 3*(parIdx == 0) + (p+1)*(5/40)
            phi_avg = x_mean

            max_low = poly3( p_low, *max_par[i][sector][0])        
            max_mid = poly3( p_mid, *max_par[i][sector][0])        
            max_high = poly3( p_high, *max_par[i][sector][0])        

            theta_min_params = [el_theta_min_params,pip_theta_min_params,pim_theta_min_params]
            params = [ e_params, pip_params, pim_params]
            
            low_loose, up_loose = returnMap(phi_range, p_low,
                                            theta_min_params[i], 
                                            params[i], sector, 
                                            expo(p_low, *(mean_par[i][sector][0])), 
                                            parIdx, max_low)
            low_mid, up_mid = returnMap(phi_range, p_mid, theta_min_params[i], params[i], sector, expo(p_mid, *(mean_par[i][sector][0])), parIdx, max_mid)
            low_tight, up_tight = returnMap(phi_range, p_high, theta_min_params[i], params[i], sector, expo(p_high, *(mean_par[i][sector][0])), parIdx, max_high)
            
            ax[pan].text( max_mid, expo(p_mid, *(mean_par[i][sector][0])), f"Sector {sector+1}", color='Green')


            side=-1
            ax[pan].plot( phi_range, low_loose,color='red')
            ax[pan].plot( phi_range, low_mid,color='white')
            ax[pan].plot( phi_range, low_tight,color='cyan')
            
            ax[pan].plot( phi_range, up_loose,color='red')
            ax[pan].plot( phi_range, up_mid,color='white')
            ax[pan].plot( phi_range, up_tight,color='cyan')


            #side=1
            #ax[pan].plot( phi_range, phiFunction_fixed(phi_range, *popt_low),color='red')
        

In [ ]:
plotMaps(0)

In [ ]:
plotMaps(1)

In [ ]:
plotMaps(2)